# Blending and Stacking Regressors

**Notebook 2 of 3.** A majority vote needs models that disagree. This notebook builds five
models that are all the *same algorithm* on the *same columns*, differing only in their
settings, and asks whether a learned combiner can still find anything in them. Then it tries
three different libraries instead, and compares.

## 1. The data

`data/california_housing.csv`: 20,640 districts from the 1990 California census. Eight
numeric columns, plus `location`, a four-value column one-hot encoded here. The target
`MedHouseVal` is the district's median house value in units of **$100,000**, so error is
RMSE in those units and 0.01 is a thousand dollars.

Predicting the training mean for every district scores 1.145, which is the number any model
here has to beat by a wide margin.

In [ ]:
import time

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

from lightgbm import LGBMRegressor

RSEED = 42

# n_jobs=4 rather than -1. LightGBM's docs ask for the number of real CPU cores, not
# threads -- most CPUs run two threads per core -- and warn against large thread counts
# on a small table. -1 gives threads, so a four-core laptop would ask for eight.
# Four is a conservative floor: count your own cores and raise it
N_JOBS = 4

In [ ]:
df = pd.read_csv("data/california_housing.csv")

y = df["MedHouseVal"]
X = pd.get_dummies(df.drop(columns="MedHouseVal"), columns=["location"]).astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RSEED
)


def score(prediction):
    """RMSE of a prediction array against the test districts, in units of $100,000."""
    return root_mean_squared_error(y_test, prediction)


print(f"{len(X_train)} training districts, {len(X_test)} test districts, {X.shape[1]} columns")
print(f"predicting the training mean everywhere: {score(np.full(len(y_test), y_train.mean())):.4f}")

## 2. Five models from one algorithm

Every member below is an `LGBMRegressor` on the same twelve columns. Only the settings
differ, and each setting is one of the controls that decides how a boosted tree fits:

- **balanced** — 63 leaves, the middle of the range;
- **shallow** — 15 leaves, too small for a table this size and deliberately so;
- **deep** — 127 leaves with `min_child_samples=5`, free to chase individual districts;
- **sampled** — 63 leaves but each tree sees only half the columns, so it cannot lean on
  the same feature every round;
- **huber** — 63 leaves fitting the **Huber loss**, which grows linearly rather than
  quadratically once an error passes a threshold, so extreme districts pull it less than
  they pull the squared-error members.

The last two are the only ones that change *what the model is fitted to* rather than how
large it is allowed to grow.

In [ ]:
# 150 trees at a slightly higher learning rate: enough to separate the five settings
# without spending a minute on every fit
LGB = dict(n_estimators=150, learning_rate=0.08, random_state=RSEED,
           n_jobs=N_JOBS, verbose=-1)


def five_lightgbms():
    """Build a fresh, unfitted dictionary of the five members.

    A function rather than a dictionary, because each section below fits them on
    different rows and must not inherit another section's fitted state.

    Returns:
        dict: name -> unfitted LGBMRegressor.
    """
    return {
        "balanced": LGBMRegressor(num_leaves=63, min_child_samples=10, **LGB),
        "shallow": LGBMRegressor(num_leaves=15, min_child_samples=20, **LGB),
        "deep": LGBMRegressor(num_leaves=127, min_child_samples=5, **LGB),
        "sampled": LGBMRegressor(num_leaves=63, colsample_bytree=0.5,
                                 min_child_samples=10, **LGB),
        "huber": LGBMRegressor(num_leaves=63, objective="huber",
                               min_child_samples=10, **LGB),
    }


NAMES = list(five_lightgbms())

In [ ]:
# fit all five on the whole training set: this is the bar any combination has to clear,
# and these predictions are reused as the meta-model's test columns in section 5
members = five_lightgbms()
test_prediction = {}

start = time.perf_counter()
for name, model in members.items():
    model.fit(X_train, y_train)
    test_prediction[name] = model.predict(X_test)
    print(f"  {name:<10} {score(test_prediction[name]):.4f}")

# P has one row per member and one column per test district
P = np.stack([test_prediction[name] for name in NAMES])
best = min(NAMES, key=lambda n: score(test_prediction[n]))
print(f"\nbest member: {best}, {score(test_prediction[best]):.4f}   "
      f"[{time.perf_counter() - start:.0f}s for all five]")

**0.4383 for `sampled`, 0.4755 for `shallow`** — a spread of 0.037, which is $3,700 of
median house value. The five are not equally good, and that matters for what follows.

## 3. The average does not work

The obvious combination is the mean of the five predictions. It is also the one that fails
here, and the reason is visible in two numbers.

In [ ]:
print(f"plain average      {score(P.mean(axis=0)):.4f}")
print(f"best member        {score(test_prediction[best]):.4f}")

# the error each member makes on each district; correlated errors cannot cancel
errors = P - y_test.values
correlation = np.corrcoef(errors)
off_diagonal = correlation[np.triu_indices(len(NAMES), k=1)]
print(f"\nerror correlation between members: {off_diagonal.min():.3f} to {off_diagonal.max():.3f}")

**The average scores 0.4396, worse than `sampled` alone at 0.4383.** Averaging made things
worse, and two facts explain it:

- **the members are not equally good** — a mean gives `shallow`, at 0.4755, exactly the same
  say as `sampled` at 0.4383;
- **their errors are correlated 0.910 to 0.969** — they are the same algorithm on the same
  columns, so they are wrong about the same districts, and errors that agree cannot cancel.

A fixed rule cannot fix either problem. It has no way to notice that one member is worse,
and no way to use the small part where they differ. Both need a combiner that is **fitted**.

## 4. Blending: a meta-model trained on held-back rows

**Blending** holds back part of the training data. The members are fitted on what is left,
they predict the held-back rows, and those predictions become the columns a **meta-model**
is fitted on. The held-back rows are the point: the members never saw them, so their
predictions there are as unflattering as they will be on genuinely new districts.

> **Exercise 1.** Using the split above, fit the five members on the base part, build a
> column per member from their predictions on the blend part, fit a `LinearRegression` on
> those columns, and score it on the test districts. Print the fitted weights, and say which
> member got the strangest one.

<details><summary>
Click here for a hint...
</summary>

The meta-model needs its columns built twice — once on the blend part to fit it, once on the
test districts to score it — and in the same order both times. `np.column_stack([...])` over
`NAMES` guarantees that.

</details>

In [ ]:
# three quarters of the training districts to fit the members, one quarter held back
# for the meta-model to learn on. Given here so the exercise starts from the same split
X_base, X_blend, y_base, y_blend = train_test_split(
    X_train, y_train, test_size=0.25, random_state=RSEED
)
print(f"{len(X_base)} districts to fit the members, {len(X_blend)} for the meta-model")

In [ ]:
# your code: the five members fitted on the base part, a LinearRegression fitted on
# their blend-part predictions, and the result scored on the test districts

<details><summary>
Click here to compare your answer...
</summary>

| | test RMSE |
| :-- | --: |
| best member (`sampled`) | 0.4383 |
| plain average | 0.4396 |
| **blending** | **0.4345** |

Weights: `balanced` 0.100, `shallow` **−0.273**, `deep` 0.117, `sampled` 0.720, `huber`
0.350.

**`shallow` gets a negative weight.** The meta-model is not ignoring the weakest member, it
is **subtracting** it: wherever `shallow` runs high, the combination is pushed down. A model
that is reliably wrong in a particular direction carries usable information, and no fixed
rule can reach it — an average can only ever add.

That is the whole gap between 0.4396 and 0.4345.

</details>

## 5. Stacking: the same idea without the data cost

Blending has a price, printed by the cell above: the members were fitted on **12,384
districts instead of 16,512**. A quarter of the training data went to the meta-model rather
than to the models it combines.

**Stacking** removes that price with cross-validation. Split the training rows into folds;
for each fold, fit the members on the others and predict the held-out one. Every training
row ends up with a prediction from a member that did not see it — **out-of-fold**
predictions — so the meta-model gets a full-length column and the members can then be refit
on everything.

In [ ]:
# three folds rather than five: three is enough to build the columns and costs 40% less
cv = KFold(3, shuffle=True, random_state=RSEED)

start = time.perf_counter()
# cross_val_predict returns one prediction per training row, each made by a copy of the
# model that was fitted without that row
oof_columns = np.column_stack([
    cross_val_predict(model, X_train, y_train, cv=cv)
    for model in five_lightgbms().values()
])

stack_meta = LinearRegression()
stack_meta.fit(oof_columns, y_train)

# the test columns are section 2's predictions, from members fitted on ALL the training
# districts: nothing was held back, which is the saving over blending
print(f"stacking   {score(stack_meta.predict(P.T)):.4f}   [{time.perf_counter() - start:.0f}s]")
print("\nweights:", dict(zip(NAMES, stack_meta.coef_.round(3))))

**0.4310, against blending's 0.4345 and the best member's 0.4383.**

The weights tell the same story as blending's — `sampled` 0.649, `shallow` **−0.180** — so
both combiners learned the same thing. Stacking is ahead because its members were fitted on
all 16,512 districts rather than 12,384, not because it learned anything cleverer.

The trade, stated plainly:

- **blending** fits each member once, and pays for it with a quarter of the training data;
- **stacking** fits each member four times here, three folds plus a refit, and pays nothing
  in data.

On a large table, buy the cheaper one. On a small table, data is the scarcer resource and
stacking is worth the extra fits.

## Summary

- Five models of one algorithm on one set of columns have errors correlated **0.910 to
  0.969**, and their plain average scored **0.4396 — worse than the best of them**, 0.4383.
- A **fitted** combiner found something the average could not: **blending 0.4345**,
  **stacking 0.4310**. Both gave the weakest member a **negative** weight, subtracting it
  rather than ignoring it, which no fixed rule can do.
- **Blending** holds back a quarter of the training rows and fits each member once;
  **stacking** builds the same columns out-of-fold and refits on everything. Stacking costs
  more fits and no data.
- The gap between the two is **data, not cleverness**: both combiners learned the same
  weights, and stacking won because its members were fitted on all 16,512 districts.

## References & Further Reading

- [**scikit-learn: Stacked generalization**](https://scikit-learn.org/stable/modules/ensemble.html#stacked-generalization): `StackingRegressor`, which wraps section 5 into one estimator and refits the members for you.
- [**scikit-learn: cross_val_predict**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html): How the out-of-fold columns are produced, and the warning about using it as a performance measure.
- [**Wolpert, Stacked Generalization (1992)**](https://doi.org/10.1016/S0893-6080(05)80023-1): The original, where the out-of-fold requirement is the central point rather than a detail.
- [**LightGBM: Parameters**](https://lightgbm.readthedocs.io/en/stable/Parameters.html): `num_leaves`, `min_child_samples`, `colsample_bytree` and the `objective` values, including `huber`.
- [**California housing**](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset): Block groups from the 1990 US census, derived by Pace and Barry (1997). **Modified here:** scikit-learn's eight numeric columns and target, with a `location` column added from Aurélien Géron's edition.